# ゼロから作る Deep Learning ❸ 輪読会
## 第2ステージ「自然なコードで表現する」 ― ステップ 18 〜 24

### これまでの内容

ステップ 11〜17 では、DeZero の計算グラフを複数入出力・分岐・合流に対応させた。

- `Variable.backward()` は関数の世代 (`generation`) が大きい順に逆伝播する

- 同じ変数へ届いた微分は加算し、`cleargrad()` でリセットできる

- `Function.outputs` を弱参照にして、`Variable` と `Function` の循環参照を解消した

この時点で複雑な計算グラフも正しく微分できる。しかし、学習を繰り返すにはメモリ効率と使い勝手がまだ十分ではない。

### このノートブックの目標

ステップ 18〜24 では、計算グラフの仕組みを保ったまま、普段の NumPy コードに近い書き方へ仕上げる。

| ステップ | テーマ | ゴール |
|---|---|---|
| 18 | メモリ使用量を減らすモード | 逆伝播済みの関数出力の勾配を破棄し、微分が不要なときは計算グラフ自体を作らない |
| 19 | 変数を使いやすく | 名前・形状などの属性と読みやすい表示を追加 |
| 20 | 演算子のオーバーロード（1） | `a * b + c` と書けるようにする |
| 21 | 演算子のオーバーロード（2） | NumPy 配列や Python の数値と一緒に計算する |
| 22 | 演算子のオーバーロード（3） | 負号・減算・除算・累乗を追加 |
| 23 | パッケージとしてまとめる | `dezero` パッケージから機能を読み込む |
| 24 | 複雑な関数の微分 | 長い数式も通常の Python 式のまま微分する |

最後には、たとえば次の式をそのまま計算グラフとして扱えるようになる。

```python
z = 0.26 * (x ** 2 + y ** 2) - 0.48 * x * y
z.backward()
```

## ライブラリ読み込み

`weakref` は計算グラフの循環参照を避けるため、`contextlib` は一時的なモード切り替えを `with` 文で安全に扱うために使う。

In [ ]:
import contextlib
import weakref

import numpy as np

## ステップ 17 時点の実装

このノートブックを単独で上から実行できるように、ステップ 17 の到達点を一度だけ定義する。以降は、各ステップで変更する部分だけを差し替えていく。

現状では、逆伝播に使ったすべての変数の `grad` が残り、順伝播では常に計算グラフを作る。

In [ ]:
def as_array(x):
    if np.isscalar(x):
        return np.array(x)
    return x


class Variable:
    def __init__(self, data):
        if data is not None and not isinstance(data, np.ndarray):
            raise TypeError(f"{type(data)} is not supported")

        self.data = data
        self.grad = None
        self.creator = None
        self.generation = 0

    def set_creator(self, func):
        self.creator = func
        self.generation = func.generation + 1

    def cleargrad(self):
        self.grad = None

    def backward(self):
        if self.grad is None:
            self.grad = np.ones_like(self.data)

        funcs = []
        seen_set = set()

        def add_func(f):
            if f not in seen_set:
                funcs.append(f)
                seen_set.add(f)
                funcs.sort(key=lambda func: func.generation)

        add_func(self.creator)

        while funcs:
            f = funcs.pop()
            gys = [output().grad for output in f.outputs]
            gxs = f.backward(*gys)
            if not isinstance(gxs, tuple):
                gxs = (gxs,)

            for x, gx in zip(f.inputs, gxs):
                if x.grad is None:
                    x.grad = gx
                else:
                    x.grad = x.grad + gx

                if x.creator is not None:
                    add_func(x.creator)


class Function:
    def __call__(self, *inputs):
        xs = [x.data for x in inputs]
        ys = self.forward(*xs)
        if not isinstance(ys, tuple):
            ys = (ys,)
        outputs = [Variable(as_array(y)) for y in ys]

        self.generation = max(x.generation for x in inputs)
        for output in outputs:
            output.set_creator(self)
        self.inputs = inputs
        self.outputs = [weakref.ref(output) for output in outputs]

        return outputs if len(outputs) > 1 else outputs[0]

    def forward(self, *xs):
        raise NotImplementedError

    def backward(self, *gys):
        raise NotImplementedError


class Square(Function):
    def forward(self, x):
        return x ** 2

    def backward(self, gy):
        x = self.inputs[0].data
        return 2 * x * gy


def square(x):
    return Square()(x)


class Add(Function):
    def forward(self, x0, x1):
        return x0 + x1

    def backward(self, gy):
        return gy, gy


def add(x0, x1):
    return Add()(x0, x1)

In [ ]:
# 分岐・合流を含むグラフでも、世代順に正しく逆伝播できる
x = Variable(np.array(2.0))
a = square(x)
y = add(square(a), square(a))  # y = 2x^4
y.backward()

print("y      =", y.data, "（期待値 32.0）")
print("x.grad =", x.grad, "（期待値 64.0）")

---
# ステップ 18：メモリ使用量を減らすモード

ニューラルネットワークでは大きな配列を何層も通すため、計算途中のデータを必要以上に保持するとメモリを圧迫する。このステップでは、用途の異なる 2 種類の節約を行う。

1. 逆伝播済みの関数出力の勾配（最終出力を含む）を捨てる
2. 推論など微分が不要な処理では、計算グラフを作らない

## 18.1 不要な微分は保持しない

現状の `backward()` は、葉の変数だけでなく途中の変数と最終出力にも `grad` を残す。次のグラフで確認する。

```python
t = add(x0, x1)
y = add(x0, t)
```

In [ ]:
x0 = Variable(np.array(1.0))
x1 = Variable(np.array(1.0))
t = add(x0, x1)
y = add(x0, t)
y.backward()

print("y.grad, t.grad   =", y.grad, t.grad)
print("x0.grad, x1.grad =", x0.grad, x1.grad)

通常、保持したいのは葉の変数、とくに学習対象パラメータの勾配である。出力側の `y.grad` と途中の `t.grad` は、それぞれの関数の逆伝播が終われば不要になる。

`backward(retain_grad=False)` を既定値とし、関数を処理した直後に、その関数の出力の勾配を `None` に戻す。途中の勾配を調査したい場合だけ `retain_grad=True` を指定する。

In [ ]:
class Variable:
    def __init__(self, data):
        if data is not None and not isinstance(data, np.ndarray):
            raise TypeError(f"{type(data)} is not supported")

        self.data = data
        self.grad = None
        self.creator = None
        self.generation = 0

    def set_creator(self, func):
        self.creator = func
        self.generation = func.generation + 1

    def cleargrad(self):
        self.grad = None

    def backward(self, retain_grad=False):
        if self.grad is None:
            self.grad = np.ones_like(self.data)

        funcs = []
        seen_set = set()

        def add_func(f):
            if f not in seen_set:
                funcs.append(f)
                seen_set.add(f)
                funcs.sort(key=lambda func: func.generation)

        add_func(self.creator)

        while funcs:
            f = funcs.pop()
            gys = [output().grad for output in f.outputs]
            gxs = f.backward(*gys)
            if not isinstance(gxs, tuple):
                gxs = (gxs,)

            for x, gx in zip(f.inputs, gxs):
                if x.grad is None:
                    x.grad = gx
                else:
                    x.grad = x.grad + gx

                if x.creator is not None:
                    add_func(x.creator)

            if not retain_grad:
                for y in f.outputs:
                    y().grad = None

In [ ]:
def run_retain_grad_example(retain_grad):
    x0 = Variable(np.array(1.0))
    x1 = Variable(np.array(1.0))
    t = add(x0, x1)
    y = add(x0, t)
    y.backward(retain_grad=retain_grad)
    grads = (y.grad, t.grad, x0.grad, x1.grad)
    return tuple(None if grad is None else grad.item() for grad in grads)


print("既定値 False:", run_retain_grad_example(False))
print("明示的に True:", run_retain_grad_example(True))

`False` の場合は最終出力を含む関数出力の勾配が削除され、葉の `x0.grad=2.0` と `x1.grad=1.0` だけが残る。`True` なら途中の勾配も保持できる。

## 18.2 `Function` クラスの復習

順伝播で計算グラフを作っているのは `Function.__call__` の次の処理である。

- 出力の `creator` に関数自身を設定する
- 関数が `inputs` と `outputs` を保持する
- 入力の最大世代を関数の `generation` に設定する

逆伝播が不要なら、この記録を省くことでグラフのオブジェクトと参照を作らずに済む。

## 18.3 `Config` クラスによる切り替え

フレームワーク全体で共有する設定 `Config.enable_backprop` を用意する。`True` なら通常どおり計算グラフを作り、`False` なら順伝播の値だけを返す。

In [ ]:
class Config:
    enable_backprop = True

## 18.4 モードを切り替える

`Function.__call__` のグラフ構築部分を `if Config.enable_backprop:` の中へ移す。`False` のときに得られる出力は `creator=None` となり、過去の計算へさかのぼれない。

In [ ]:
class Function:
    def __call__(self, *inputs):
        xs = [x.data for x in inputs]
        ys = self.forward(*xs)
        if not isinstance(ys, tuple):
            ys = (ys,)
        outputs = [Variable(as_array(y)) for y in ys]

        if Config.enable_backprop:
            self.generation = max(x.generation for x in inputs)
            for output in outputs:
                output.set_creator(self)
            self.inputs = inputs
            self.outputs = [weakref.ref(output) for output in outputs]

        return outputs if len(outputs) > 1 else outputs[0]

    def forward(self, *xs):
        raise NotImplementedError

    def backward(self, *gys):
        raise NotImplementedError


class Square(Function):
    def forward(self, x):
        return x ** 2

    def backward(self, gy):
        x = self.inputs[0].data
        return 2 * x * gy


def square(x):
    return Square()(x)


class Add(Function):
    def forward(self, x0, x1):
        return x0 + x1

    def backward(self, gy):
        return gy, gy


def add(x0, x1):
    return Add()(x0, x1)

## 18.5 `with` 文による切り替え

設定を手作業で `False` にしてから `True` に戻す方法は、途中で例外が起きると元に戻せない。コンテキストマネージャ `using_config` を使い、`with` ブロックを抜けると必ず以前の値へ復元する。

`no_grad()` は、よく使う `using_config('enable_backprop', False)` に名前を付けた短縮形である。「勾配を削除する」のではなく、「計算グラフを作らない」機能である点に注意する。

In [ ]:
@contextlib.contextmanager
def using_config(name, value):
    old_value = getattr(Config, name)
    setattr(Config, name, value)
    try:
        yield
    finally:
        setattr(Config, name, old_value)


def no_grad():
    return using_config("enable_backprop", False)

In [ ]:
x = Variable(np.array(2.0))

with no_grad():
    y = square(x)
    print("with の内側:", Config.enable_backprop, y.creator)

z = square(x)
print("with の外側:", Config.enable_backprop, type(z.creator).__name__)

# 例外が起きても finally により設定は復元される
try:
    with using_config("enable_backprop", False):
        raise RuntimeError("動作確認用")
except RuntimeError:
    pass

print("例外の後も復元:", Config.enable_backprop)

> **ステップ 18 のまとめ**
>
> - `backward(retain_grad=False)` は、逆伝播済みの関数出力の勾配を削除する。
> - `retain_grad=True` を指定すれば、デバッグなどのために途中の勾配を保持できる。
> - `Config.enable_backprop=False` では計算グラフ自体を作らない。
> - `using_config` と `no_grad()` により、設定を `with` 文で安全に一時変更できる。

---
# ステップ 19：変数を使いやすく

`Variable` は NumPy の `ndarray` を包むクラスである。中身を毎回 `x.data.shape` のようにたどらなくても済むよう、よく使う属性を `Variable` から直接参照できるようにする。

## 19.1 変数に名前を付ける

`name` は計算結果には影響しないが、後で計算グラフを可視化するときに変数を識別しやすくする。

## 19.2 `ndarray` の属性を引き継ぐ

| 属性 | 意味 | 例 |
|---|---|---|
| `shape` | 各軸の大きさ | `(2, 3)` |
| `ndim` | 次元数 | `2` |
| `size` | 全要素数 | `6` |
| `dtype` | データ型 | `int64` など |

これらは新しい値を保存せず、`@property` から `self.data` の属性を返す。

In [ ]:
class Variable:
    def __init__(self, data, name=None):
        if data is not None and not isinstance(data, np.ndarray):
            raise TypeError(f"{type(data)} is not supported")

        self.data = data
        self.name = name
        self.grad = None
        self.creator = None
        self.generation = 0

    @property
    def shape(self):
        return self.data.shape

    @property
    def ndim(self):
        return self.data.ndim

    @property
    def size(self):
        return self.data.size

    @property
    def dtype(self):
        return self.data.dtype

    def __len__(self):
        return len(self.data)

    def __repr__(self):
        if self.data is None:
            return "variable(None)"
        p = str(self.data).replace("\n", "\n" + " " * 9)
        return "variable(" + p + ")"

    def set_creator(self, func):
        self.creator = func
        self.generation = func.generation + 1

    def cleargrad(self):
        self.grad = None

    def backward(self, retain_grad=False):
        if self.grad is None:
            self.grad = np.ones_like(self.data)

        funcs = []
        seen_set = set()

        def add_func(f):
            if f not in seen_set:
                funcs.append(f)
                seen_set.add(f)
                funcs.sort(key=lambda func: func.generation)

        add_func(self.creator)

        while funcs:
            f = funcs.pop()
            gys = [output().grad for output in f.outputs]
            gxs = f.backward(*gys)
            if not isinstance(gxs, tuple):
                gxs = (gxs,)

            for x, gx in zip(f.inputs, gxs):
                if x.grad is None:
                    x.grad = gx
                else:
                    x.grad = x.grad + gx

                if x.creator is not None:
                    add_func(x.creator)

            if not retain_grad:
                for y in f.outputs:
                    y().grad = None

In [ ]:
x = Variable(
    np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float64),
    name="x",
)

print("name :", x.name)
print("shape:", x.shape)
print("ndim :", x.ndim)
print("size :", x.size)
print("dtype:", x.dtype)

## 19.3 `len` 関数と `print` 関数

`__len__` は `len(x)` を `len(x.data)` へつなぎ、先頭軸の大きさを返す。

`__repr__` を実装すると、`print(x)` やノートブックのセル末尾で中身を確認しやすくなる。複数行の配列では、2 行目以降を `variable(` の長さだけ字下げして括弧を揃える。データがない `Variable(None)` も安全に表示する。

In [ ]:
print("len(x) =", len(x))
print(x)
print(Variable(np.array(3.0)))
print(Variable(None))

> **ステップ 19 のまとめ**
>
> - 任意の `name` を持たせ、計算グラフ上で変数を識別できるようにした。
> - `shape`、`ndim`、`size`、`dtype`、`len` を `Variable` から直接使えるようにした。
> - `__repr__` により、スカラー・多次元配列・`None` を読みやすく表示できる。

---
# ステップ 20：演算子のオーバーロード（1）

ここまでは足し算を `add(a, b)`、二乗を `square(x)` と書いてきた。Python の特殊メソッドを `Variable` に登録すると、通常の数式と同じ演算子で `Function` を呼び出せる。

## 20.1 乗算の実装

$y=x_0x_1$ の偏微分は次のとおりである。逆伝播では上流から来た微分 $g_y$ を掛ける。

$$
\frac{\partial y}{\partial x_0}=x_1,\qquad
\frac{\partial y}{\partial x_1}=x_0
$$

したがって `Mul.backward` は `(gy * x1, gy * x0)` を返す。

In [ ]:
class Mul(Function):
    def forward(self, x0, x1):
        return x0 * x1

    def backward(self, gy):
        x0 = self.inputs[0].data
        x1 = self.inputs[1].data
        return gy * x1, gy * x0


def mul(x0, x1):
    return Mul()(x0, x1)

## 20.2 特殊メソッドへの登録

`a * b` を評価すると Python は `a.__mul__(b)` を、`a + b` なら `a.__add__(b)` を呼ぶ。そこで既存の `mul` と `add` をそれぞれの特殊メソッドへ登録する。

```text
a * b + c
  │     │
  Mul   Add
  └─ どちらも Function として計算グラフへ記録される
```

In [ ]:
Variable.__mul__ = mul
Variable.__add__ = add

a = Variable(np.array(3.0), name="a")
b = Variable(np.array(2.0), name="b")
c = Variable(np.array(1.0), name="c")

y = a * b + c
print("y =", y)

In [ ]:
y.backward()

print("a.grad =", a.grad, "（期待値 b = 2.0）")
print("b.grad =", b.grad, "（期待値 a = 3.0）")
print("c.grad =", c.grad, "（期待値 1.0）")

# 同じ変数を 2 回使う場合も、ステップ 14 の勾配加算が働く
x = Variable(np.array(3.0))
y = x * x
y.backward()
print("d(x*x)/dx =", x.grad, "（期待値 6.0）")

> **ステップ 20 のまとめ**
>
> - 積の順伝播と逆伝播を行う `Mul` を実装した。
> - `Variable.__mul__` と `Variable.__add__` に関数を登録し、`a * b + c` と書けるようにした。
> - 演算子で書いても、内部ではこれまでと同じ `Function` が計算グラフを作る。
> - この時点では演算相手も `Variable` である必要があり、`x + 3.0` はまだ扱えない。

---
# ステップ 21：演算子のオーバーロード（2）

実際の数式では `Variable` 同士だけでなく、NumPy 配列や Python の `float`・`int` も一緒に使う。まず、ステップ 20 の実装で定数との足し算が失敗することを安全に確認する。

## 21.1 `Variable` と定数を一緒に使う

In [ ]:
x = Variable(np.array(2.0))

try:
    x + 3.0
except (AttributeError, TypeError) as error:
    print(type(error).__name__, ":", error)

`Function.__call__` はすべての入力に `.data` があると仮定していたため、数値を直接受け取れない。

ここでは 2 つの変換を役割分担させる。

- `as_array(x)`：Python / NumPy のスカラーを 0 次元 `ndarray` に揃える
- `as_variable(obj)`：`Variable` でなければ `Variable(obj)` で包む

演算関数の入口でスカラーを `ndarray` にし、`Function.__call__` ですべての入力を `Variable` に揃える。

In [ ]:
def as_variable(obj):
    if isinstance(obj, Variable):
        return obj
    return Variable(obj)


class Function:
    def __call__(self, *inputs):
        inputs = [as_variable(x) for x in inputs]

        xs = [x.data for x in inputs]
        ys = self.forward(*xs)
        if not isinstance(ys, tuple):
            ys = (ys,)
        outputs = [Variable(as_array(y)) for y in ys]

        if Config.enable_backprop:
            self.generation = max(x.generation for x in inputs)
            for output in outputs:
                output.set_creator(self)
            self.inputs = inputs
            self.outputs = [weakref.ref(output) for output in outputs]

        return outputs if len(outputs) > 1 else outputs[0]

    def forward(self, *xs):
        raise NotImplementedError

    def backward(self, *gys):
        raise NotImplementedError


class Square(Function):
    def forward(self, x):
        return x ** 2

    def backward(self, gy):
        x = self.inputs[0].data
        return 2 * x * gy


def square(x):
    return Square()(x)


class Add(Function):
    def forward(self, x0, x1):
        return x0 + x1

    def backward(self, gy):
        return gy, gy


def add(x0, x1):
    x1 = as_array(x1)
    return Add()(x0, x1)


class Mul(Function):
    def forward(self, x0, x1):
        return x0 * x1

    def backward(self, gy):
        x0 = self.inputs[0].data
        x1 = self.inputs[1].data
        return gy * x1, gy * x0


def mul(x0, x1):
    x1 = as_array(x1)
    return Mul()(x0, x1)

## 21.2 左右どちらからでも計算できるようにする

`x + 3` は `x.__add__(3)` だが、`3 + x` では左側の数値が先に処理を試みる。左側で計算できなかったとき、Python は右側の逆演算 `x.__radd__(3)` を呼ぶ。乗算も同様に `__rmul__` が必要である。

NumPy 配列が左側にある場合は、NumPy が先に要素ごとの処理を始めないよう、`Variable.__array_priority__` を高くする。これにより `np.array(3.0) + x` も `Variable` 側の演算へ渡る。

> このステップはスカラーとの混在に対応する段階である。形状の異なる配列をブロードキャストしたときの勾配調整は、後のステップ 40 で扱う。

In [ ]:
Variable.__array_priority__ = 200
Variable.__add__ = add
Variable.__radd__ = add
Variable.__mul__ = mul
Variable.__rmul__ = mul

x = Variable(np.array(2.0))

print("x + ndarray      =", x + np.array(3.0))
print("x + float        =", x + 3.0)
print("float + x        =", 3.0 + x)
print("ndarray + x      =", np.array(3.0) + x)
print("3.0 * x + 1.0    =", 3.0 * x + 1.0)

In [ ]:
x = Variable(np.array(2.0))
y = 3.0 * x + 1.0
y.backward()

print("y      =", y.data, "（期待値 7.0）")
print("x.grad =", x.grad, "（期待値 3.0）")

> **ステップ 21 のまとめ**
>
> - `as_variable` と `as_array` で、入力を `Variable` と `ndarray` に揃えた。
> - `__radd__`・`__rmul__` により、定数が左側にある式も扱える。
> - `__array_priority__` により、NumPy 配列が左側でも `Variable` の演算を優先できる。
> - `3.0 * x + 1.0` のような式も計算グラフになり、定数を含めて正しく逆伝播できる。

---
# ステップ 22：演算子のオーバーロード（3）

残りの基本演算を追加する。演算子と特殊メソッドの対応は次のとおり。

| 式 | 通常演算 | 逆演算 |
|---|---|---|
| `-x` | `__neg__` | — |
| `x - c`, `c - x` | `__sub__` | `__rsub__` |
| `x / c`, `c / x` | `__truediv__` | `__rtruediv__` |
| `x ** c` | `__pow__` | — |

## 22.1 負号

$y=-x$ の微分は $-1$ なので、逆伝播では上流の微分の符号を反転する。

In [ ]:
class Neg(Function):
    def forward(self, x):
        return -x

    def backward(self, gy):
        return -gy


def neg(x):
    return Neg()(x)


Variable.__neg__ = neg

x = Variable(np.array(2.0))
y = -x
y.backward()
print("y =", y, ", x.grad =", x.grad, "（期待値 -1.0）")

## 22.2 減算

$y=x_0-x_1$ の偏微分は $(1,-1)$ である。`c - x` は順序を入れ替えて `sub(c, x)` とする必要があるため、`rsub` を別に用意する。

In [ ]:
class Sub(Function):
    def forward(self, x0, x1):
        return x0 - x1

    def backward(self, gy):
        return gy, -gy


def sub(x0, x1):
    x1 = as_array(x1)
    return Sub()(x0, x1)


def rsub(x0, x1):
    x1 = as_array(x1)
    return Sub()(x1, x0)


Variable.__sub__ = sub
Variable.__rsub__ = rsub

x = Variable(np.array(2.0))
y = 2.0 - x
y.backward()
print("2.0 - x =", y, ", x.grad =", x.grad, "（期待値 -1.0）")

x = Variable(np.array(2.0))
y = x - 1.0
y.backward()
print("x - 1.0 =", y, ", x.grad =", x.grad, "（期待値 1.0）")

## 22.3 除算

$y=x_0/x_1$ の偏微分は次のとおりである。

$$
\frac{\partial y}{\partial x_0}=\frac{1}{x_1},\qquad
\frac{\partial y}{\partial x_1}=-\frac{x_0}{x_1^2}
$$

減算と同じく、`c / x` ではオペランドの順序が重要なので `rdiv` で入れ替える。

In [ ]:
class Div(Function):
    def forward(self, x0, x1):
        return x0 / x1

    def backward(self, gy):
        x0 = self.inputs[0].data
        x1 = self.inputs[1].data
        gx0 = gy / x1
        gx1 = gy * (-x0 / x1 ** 2)
        return gx0, gx1


def div(x0, x1):
    x1 = as_array(x1)
    return Div()(x0, x1)


def rdiv(x0, x1):
    x1 = as_array(x1)
    return Div()(x1, x0)


Variable.__truediv__ = div
Variable.__rtruediv__ = rdiv

x = Variable(np.array(2.0))
y = 3.0 / x
y.backward()
print("3.0 / x =", y, ", x.grad =", x.grad, "（期待値 -0.75）")

x = Variable(np.array(2.0))
y = x / 2.0
y.backward()
print("x / 2.0 =", y, ", x.grad =", x.grad, "（期待値 0.5）")

## 22.4 累乗

指数 $c$ を定数とした $y=x^c$ の微分は $cx^{c-1}$ である。`Pow` のインスタンスが `c` を保持し、逆伝播で使う。

この実装は底 `x` を微分するもので、指数 `c` 自体を `Variable` として微分する機能ではない。

In [ ]:
class Pow(Function):
    def __init__(self, c):
        self.c = c

    def forward(self, x):
        return x ** self.c

    def backward(self, gy):
        x = self.inputs[0].data
        return self.c * x ** (self.c - 1) * gy


def pow(x, c):
    return Pow(c)(x)


Variable.__pow__ = pow

x = Variable(np.array(2.0))
y = x ** 3
y.backward()
print("x ** 3 =", y, ", x.grad =", x.grad, "（期待値 12.0）")

最後に、追加した演算を 1 本の式で組み合わせる。

$$
y=\frac{2-x}{x^2}
$$

$x=2$ では $y=0$、微分は $-0.25$ になる。

In [ ]:
x = Variable(np.array(2.0))
y = (2.0 - x) / (x ** 2)
y.backward()

print("y      =", y.data, "（期待値 0.0）")
print("x.grad =", x.grad, "（期待値 -0.25）")

> **ステップ 22 のまとめ**
>
> - 負号 `-`、減算 `-`、除算 `/`、累乗 `**` を `Function` として実装した。
> - 順序が意味を持つ減算・除算では、`__rsub__` と `__rtruediv__` で入力を入れ替えた。
> - 各演算の順伝播だけでなく、解析的な期待値と逆伝播の結果も確認した。
> - これで基本的な数式を通常の Python 演算子だけで組み立てられる。

---
# ステップ 23：パッケージとしてまとめる

ここまでは 1 つのノートブック上でクラスと関数を定義してきた。再利用できるライブラリにするには、実装を Python パッケージへ分け、利用者が公開 API だけを import できるようにする。

## 23.1 ファイル構成

この段階の中心は次の 2 ファイルである。

```text
dezero/
├── __init__.py      # 外部へ公開する名前をまとめる
└── core_simple.py   # ステップ 22 までの Variable / Function / 演算
```

`core_simple.py` の `setup_variable()` は、`Variable.__add__` などの特殊メソッドをまとめて登録する。`__init__.py` から必要な名前を再公開すれば、利用者は内部のファイル構成を意識せず次のように書ける。

```python
from dezero import Variable
```

最小構成では、2 ファイルの接続は次のようになる。

```python
# dezero/core_simple.py
def setup_variable():
    Variable.__add__ = add
    Variable.__radd__ = add
    Variable.__mul__ = mul
    # ...残りの演算子も登録...

# dezero/__init__.py（ステップ 23 時点）
from dezero.core_simple import Variable, no_grad, setup_variable
setup_variable()
```

> このリポジトリはステップ 60 まで完成した状態なので、現在の `dezero.Variable` は機能を拡張した `dezero.core.Variable` を指す。ここではステップ 23〜24 の挙動を再現するため、パッケージ内の `core_simple` を明示的に使う。

## 23.2 ノートブックからパッケージを読み込む

Jupyter の作業ディレクトリは実行方法によってリポジトリ直下または `notebooks/` になる。`__file__` はノートブックでは定義されないため、それに依存せず、現在位置から上位へたどって `dezero/` を探す。

In [ ]:
import sys
from pathlib import Path


current_dir = Path.cwd().resolve()
repo_root = next(
    (path for path in (current_dir, *current_dir.parents)
     if (path / "dezero").is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError("dezero ディレクトリを含むリポジトリが見つかりません")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import dezero
from dezero.core_simple import Variable as DeZeroVariable
from dezero.core_simple import no_grad as dezero_no_grad
from dezero.core_simple import setup_variable as setup_dezero_variable


setup_dezero_variable()

print("現在の公開 Variable :", dezero.Variable.__module__)
print("この節で使う Variable:", DeZeroVariable.__module__)

## 23.3 パッケージ版の動作確認

`setup_variable()` の後は、ノートブック内で作ったクラスと同じように演算子を使える。`core_simple` では勾配を別の `Variable` で包まず、NumPy 配列または NumPy スカラーとして保持するため、ステップ 22 時点の意味も保たれる。

In [ ]:
x = DeZeroVariable(np.array(1.0))
y = (x + 3.0) ** 2
y.backward()

print("y      =", y)
print("x.grad =", x.grad, "（期待値 8.0）")

with dezero_no_grad():
    inference_y = (x + 3.0) ** 2

print("no_grad では creator =", inference_y.creator)

> **ステップ 23 のまとめ**
>
> - コア実装を `dezero/core_simple.py` に置き、パッケージとして再利用できる形にした。
> - `setup_variable()` に演算子の登録をまとめた。
> - `__init__.py` は内部実装から公開 API を切り離す窓口になる。
> - ノートブックでは `__file__` に依存せず、実行位置の違いに耐える import 方法にした。
> - 完成済みリポジトリの `core` と、この段階の `core_simple` の違いを明示した。

---
# ステップ 24：複雑な関数の微分

基本演算を `Function` として実装し、演算子から計算グラフを作れるようになった。その結果、複雑な式ごとに専用の `backward` を書く必要はない。

複雑な関数も `+`・`-`・`*`・`/`・`**` の組み合わせとして順伝播すれば、DeZero が連鎖律を使って各入力まで微分を伝える。

## 24.1 Sphere 関数

$$
f(x,y)=x^2+y^2
$$

偏微分は $(2x, 2y)$ なので、$(1,1)$ では関数値 `2`、勾配 `(2,2)` になる。

In [ ]:
def sphere(x, y):
    return x ** 2 + y ** 2


x = DeZeroVariable(np.array(1.0))
y = DeZeroVariable(np.array(1.0))
z = sphere(x, y)
z.backward()

print("z =", z.data)
print("(dz/dx, dz/dy) =", (x.grad, y.grad))

## 24.2 Matyas 関数

$$
f(x,y)=0.26(x^2+y^2)-0.48xy
$$

偏微分は次のとおりで、$(1,1)$ ではどちらも `0.04` になる。

$$
\frac{\partial f}{\partial x}=0.52x-0.48y,\qquad
\frac{\partial f}{\partial y}=0.52y-0.48x
$$

In [ ]:
def matyas(x, y):
    return 0.26 * (x ** 2 + y ** 2) - 0.48 * x * y


x = DeZeroVariable(np.array(1.0))
y = DeZeroVariable(np.array(1.0))
z = matyas(x, y)
z.backward()

print("z =", z.data)
print("(dz/dx, dz/dy) =", (x.grad, y.grad))

## 24.3 Goldstein–Price 関数

最後は項の多い Goldstein–Price 関数を試す。この式全体の微分公式を手で実装するのではなく、すでに実装した基本演算だけで記述する。

$$
\begin{aligned}
f(x,y)={}&\left[1+(x+y+1)^2
(19-14x+3x^2-14y+6xy+3y^2)\right]\\
&\times\left[30+(2x-3y)^2
(18-32x+12x^2+48y-36xy+27y^2)\right]
\end{aligned}
$$

$(1,1)$ では関数値 `1876`、勾配は `(-5376, 8064)` になる。

In [ ]:
def goldstein(x, y):
    first = 1 + (x + y + 1) ** 2 * (
        19 - 14 * x + 3 * x ** 2
        - 14 * y + 6 * x * y + 3 * y ** 2
    )
    second = 30 + (2 * x - 3 * y) ** 2 * (
        18 - 32 * x + 12 * x ** 2
        + 48 * y - 36 * x * y + 27 * y ** 2
    )
    return first * second


x = DeZeroVariable(np.array(1.0))
y = DeZeroVariable(np.array(1.0))
z = goldstein(x, y)
z.backward()

print("z =", z.data)
print("(dz/dx, dz/dy) =", (x.grad, y.grad))

## 補足：数値微分との照合

実装した 3 関数について、バックプロパゲーションの結果を中央差分

$$
\frac{\partial f}{\partial x}
\approx \frac{f(x+h,y)-f(x-h,y)}{2h}
$$

と照合する。数値微分は計算グラフに依存しないため、逆伝播の検算役になる。

In [ ]:
def numerical_grad_2d(f, x, y, h=1e-4):
    gx = (f(x + h, y) - f(x - h, y)) / (2 * h)
    gy = (f(x, y + h) - f(x, y - h)) / (2 * h)
    return np.array(gx), np.array(gy)


def backward_grad_2d(f, x_value, y_value):
    x = DeZeroVariable(np.array(x_value))
    y = DeZeroVariable(np.array(y_value))
    z = f(x, y)
    z.backward()
    return z.data, x.grad, y.grad


for name, function in [
    ("sphere", sphere),
    ("matyas", matyas),
    ("goldstein", goldstein),
]:
    value, gx, gy = backward_grad_2d(function, 1.0, 1.0)
    num_gx, num_gy = numerical_grad_2d(function, 1.0, 1.0)
    matched = np.allclose(
        np.array([gx, gy]),
        np.array([num_gx, num_gy]),
        rtol=1e-4,
        atol=1e-6,
    )
    print(
        f"{name:10s}",
        f"value={float(value):9.5f}",
        f"backward=({float(gx):11.6f}, {float(gy):11.6f})",
        f"numerical=({float(num_gx):11.6f}, {float(num_gy):11.6f})",
        f"一致={matched}",
    )

3 関数すべてで、逆伝播と数値微分が誤差の範囲内で一致する。Goldstein–Price のように長い式でも、各基本演算の `backward` が正しければ、DeZero が計算グラフを逆向きにたどって微分を合成できる。

> **ステップ 24 のまとめ**
>
> - 複雑な関数も、基本演算の組み合わせとしてそのまま記述できる。
> - 関数全体に専用の逆伝播を実装しなくても、連鎖律により入力の勾配が得られる。
> - Sphere・Matyas・Goldstein–Price の値と勾配を個別に確認した。
> - 数値微分との照合により、3 関数の逆伝播が正しいことを検証した。

---
# 今回のまとめ

ステップ 18〜24 で、DeZero は「複雑なグラフを微分できるコア」から「自然な数式で使える小さなライブラリ」へ進んだ。

1. **ステップ 18（メモリ節約）**  
   逆伝播後の不要な勾配を削除し、`no_grad()` では計算グラフ自体を作らないようにした。

2. **ステップ 19（`Variable` の利便性）**  
   名前、形状・次元・要素数・データ型、`len`、読みやすい表示を追加した。

3. **ステップ 20〜22（演算子のオーバーロード）**  
   四則演算・負号・累乗を通常の Python 演算子で書けるようにし、四則演算では NumPy 配列や数値を左右どちらにも置けるようにした。ブロードキャスト時の勾配調整と `2 ** x` のような逆累乗は、まだ扱わない。

4. **ステップ 23（パッケージ化）**  
   実装を `dezero` パッケージから再利用し、`setup_variable()` で演算子をまとめて登録した。

5. **ステップ 24（複雑な関数）**  
   長い式も基本演算の組み合わせだけで自動微分でき、数値微分とも一致することを確認した。

### ステップ 17 → 24 で変わった書き方

| ステップ 17 時点 | ステップ 24 時点 |
|---|---|
| `add(square(x), square(y))` | `x ** 2 + y ** 2` |
| 入力はすべて `Variable` | `Variable` と 0 次元 NumPy 配列・スカラー値を混在可能 |
| 順伝播で常にグラフを作る | `no_grad()` でグラフ構築を停止可能 |
| ノートブック内の定義を使う | `dezero` パッケージから再利用 |

次のステージでは、計算グラフの可視化と関数最適化を経て、高階微分へ進み、逆伝播の計算自体も計算グラフとして扱えるようにする。